# 02 · Preparación COCO Estados Unidos

Detector activo: **RT-DETRv2-S** con **OpenVINO Runtime nativo en Python**, checkpoint `merve/license-plates-rtdetrv2` con licencia Apache-2.0. Los datos y métricas nuevos se mantienen separados de los resultados históricos YOLO.


In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "optuna>=4,<5" "fast-plate-ocr==1.1.0" "roboflow==1.7.1" "openvino>=2025.4,<2027"

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_ROOT=Path("/content/SIRAP_USA_ECUADOR")
if not REPO_ROOT.exists(): subprocess.run(["git","clone","--depth","1","https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git",str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT); sys.path.insert(0,str(REPO_ROOT))
COUNTRY="usa"
MODEL_ID="merve/license-plates-rtdetrv2"
print("Perfil:",COUNTRY,"| RT-DETRv2-S:",MODEL_ID)


## Formato de entrenamiento

El notebook descarga European + USA License plates v1 como COCO Segmentation y usa las cajas bbox para RT-DETRv2-S; las tres etiquetas de placas se consolidan en license_plate. La exportación trae 13.143 imágenes de entrenamiento aumentadas, 386 de validación y 106 de test. El siguiente bloque crea un test final más amplio y vuelve a separar train/valid por grupos.

In [ ]:
from src.train import (
    RF_FORMAT,
    RF_PROJECT,
    RF_TITLE,
    RF_VERSION,
    RF_WORKSPACE,
    _annotation_file,
    convert_yolo_to_coco,
    download_roboflow_dataset,
)

DATA_DIR = None
USE_ROBOFLOW = True

print(
    f"Fuente Roboflow: {RF_TITLE} | "
    f"{RF_WORKSPACE}/{RF_PROJECT} | versión {RF_VERSION} | {RF_FORMAT}"
)

if USE_ROBOFLOW:
    # Configura ROBOFLOW_API_KEY en Secrets de Colab.
    DATA_DIR = str(download_roboflow_dataset("/content/datasets"))

if DATA_DIR:
    for split in ("train", "val", "test"):
        try:
            print(split, _annotation_file(DATA_DIR, split))
        except FileNotFoundError as exc:
            print(exc)


## Crear particiones y test final independiente

La versión Roboflow trae entrenamiento aumentado y un test de solo 106 imágenes. Esta celda conserva como test final las imágenes originales de validación y test (492 en la versión descargada), crea una validación nueva a partir de grupos del entrenamiento y mantiene juntas las variantes aumentadas de cada archivo fuente. También detecta solapamientos por nombre de origen y por SHA-256.

La exportación no contiene identificadores de vehículo o sesión. Se crea source_group_manifest.csv para añadirlos si los tienes; sin esos datos, la separación garantiza independencia por archivo fuente y familia de aumentos, pero no puede demostrar separación por vehículo. No uses el test para ajustar hiperparámetros ni umbrales.

In [ ]:
import csv
import hashlib
import json
import random
import shutil
from collections import defaultdict
from pathlib import Path

SOURCE_ROOT = Path(DATA_DIR).resolve()
INDEPENDENT_ROOT = SOURCE_ROOT.parent / "european-usa-license-plates-independent-coco"
GROUP_MANIFEST = SOURCE_ROOT / "source_group_manifest.csv"
VALIDATION_GROUP_FRACTION = 0.15
SPLIT_SEED = 42

def source_key_from_filename(filename):
    basename = Path(filename).name
    return basename.split(".rf.", 1)[0] if ".rf." in basename else Path(basename).stem

def resolve_source_image(source_folder, filename):
    basename = Path(filename).name
    candidates = [
        source_folder / filename,
        source_folder / basename,
        SOURCE_ROOT / filename,
        SOURCE_ROOT / basename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    matches = list(SOURCE_ROOT.rglob(basename))
    if matches:
        return matches[0].resolve()
    raise FileNotFoundError(f"No se encontró la imagen anotada: {filename}")

source_payloads = {}
source_records = {}
all_source_keys = set()

for split_name in ("train", "valid", "test"):
    annotation_path = _annotation_file(SOURCE_ROOT, "val" if split_name == "valid" else split_name)
    payload = json.loads(annotation_path.read_text(encoding="utf-8"))
    source_payloads[split_name] = payload
    annotations_by_image = defaultdict(list)
    for annotation in payload.get("annotations", []):
        annotations_by_image[int(annotation["image_id"])].append(annotation)

    records = []
    for image_record in payload.get("images", []):
        key = source_key_from_filename(image_record["file_name"])
        all_source_keys.add(key)
        records.append({
            "source_split": split_name,
            "source_folder": annotation_path.parent,
            "image": image_record,
            "annotations": annotations_by_image.get(int(image_record["id"]), []),
            "source_key": key,
        })
    source_records[split_name] = records

if not GROUP_MANIFEST.is_file():
    with GROUP_MANIFEST.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["source_key", "group_id"])
        writer.writeheader()
        for key in sorted(all_source_keys):
            writer.writerow({"source_key": key, "group_id": ""})
    print("Creé la plantilla de grupos:", GROUP_MANIFEST)

manual_groups = {}
with GROUP_MANIFEST.open(newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    if not {"source_key", "group_id"}.issubset(reader.fieldnames or []):
        raise ValueError("source_group_manifest.csv debe tener source_key y group_id.")
    for row in reader:
        key = (row.get("source_key") or "").strip()
        group_id = (row.get("group_id") or "").strip()
        if key and group_id:
            manual_groups[key] = group_id

for split_name, records in source_records.items():
    for record in records:
        group_id = manual_groups.get(record["source_key"])
        record["group_id"] = f"manual:{group_id}" if group_id else f"source:{record['source_key']}"

heldout_records = source_records["valid"] + source_records["test"]
heldout_group_ids = {record["group_id"] for record in heldout_records}
train_overlap = [
    record for record in source_records["train"]
    if record["group_id"] in heldout_group_ids
]
train_candidates = [
    record for record in source_records["train"]
    if record["group_id"] not in heldout_group_ids
]
if len(heldout_records) < 400:
    raise ValueError(f"El test combinado solo tiene {len(heldout_records)} imágenes; revisa la versión descargada.")

train_groups = defaultdict(list)
for record in train_candidates:
    train_groups[record["group_id"]].append(record)
group_ids = sorted(train_groups)
if len(group_ids) < 2:
    raise ValueError("No hay suficientes grupos de entrenamiento para separar validación.")
random.Random(SPLIT_SEED).shuffle(group_ids)
n_validation_groups = max(1, round(len(group_ids) * VALIDATION_GROUP_FRACTION))
validation_group_ids = set(group_ids[:n_validation_groups])

validation_records = []
for group_id in sorted(validation_group_ids):
    by_source_key = defaultdict(list)
    for record in train_groups[group_id]:
        by_source_key[record["source_key"]].append(record)
    # Una imagen por archivo fuente para validar; las variantes hermanas no cruzan a train.
    for key in sorted(by_source_key):
        validation_records.append(
            sorted(by_source_key[key], key=lambda item: item["image"]["file_name"])[0]
        )

training_records = [
    record for record in train_candidates
    if record["group_id"] not in validation_group_ids
]
split_records = {
    "train": training_records,
    "valid": validation_records,
    "test": heldout_records,
}

group_sets = {
    name: {record["group_id"] for record in records}
    for name, records in split_records.items()
}
for left, right in (("train", "valid"), ("train", "test"), ("valid", "test")):
    overlap = group_sets[left] & group_sets[right]
    if overlap:
        raise RuntimeError(f"Fuga de grupos entre {left} y {right}: {len(overlap)}")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

hash_owner = {}
hash_count = 0
for split_name, records in split_records.items():
    for record in records:
        image_path = resolve_source_image(record["source_folder"], record["image"]["file_name"])
        digest = sha256_file(image_path)
        previous_split = hash_owner.get(digest)
        if previous_split is not None and previous_split != split_name:
            raise RuntimeError(f"Imagen idéntica en {previous_split} y {split_name}: {image_path.name}")
        hash_owner[digest] = split_name
        hash_count += 1

marker = INDEPENDENT_ROOT / ".sirap_independent_split.json"
if INDEPENDENT_ROOT.exists():
    if not marker.is_file():
        raise FileExistsError(
            f"{INDEPENDENT_ROOT} ya existe y no parece creado por esta celda; elige otra ruta."
        )
    shutil.rmtree(INDEPENDENT_ROOT)
INDEPENDENT_ROOT.mkdir(parents=True)

source_licenses = source_payloads["train"].get("licenses", [])
split_manifest_rows = []
new_split_summary = {}

for split_name, records in split_records.items():
    split_folder = INDEPENDENT_ROOT / split_name
    split_folder.mkdir(parents=True, exist_ok=True)
    images_out = []
    annotations_out = []
    box_count = 0
    image_id = 0
    annotation_id = 0

    for record in sorted(records, key=lambda item: (item["source_split"], item["image"]["file_name"])):
        image_id += 1
        source_image = resolve_source_image(record["source_folder"], record["image"]["file_name"])
        output_name = f"{record['source_split']}__{Path(record['image']['file_name']).name}"
        destination = split_folder / output_name
        if destination.exists() or destination.is_symlink():
            destination.unlink()
        try:
            destination.symlink_to(source_image)
        except OSError:
            shutil.copy2(source_image, destination)

        image_out = dict(record["image"])
        image_out.update({
            "id": image_id,
            "file_name": output_name,
            "sirap_source_key": record["source_key"],
            "sirap_group_id": record["group_id"],
            "sirap_original_split": record["source_split"],
        })
        images_out.append(image_out)

        for source_annotation in record["annotations"]:
            bbox = [float(value) for value in source_annotation.get("bbox", [])]
            if len(bbox) != 4 or bbox[2] <= 0 or bbox[3] <= 0 or bbox[0] < 0 or bbox[1] < 0:
                raise ValueError(f"Caja COCO inválida en {output_name}: {source_annotation.get('bbox')}")
            annotation_id += 1
            annotations_out.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": 1,
                "bbox": bbox,
                "area": float(source_annotation.get("area", bbox[2] * bbox[3])),
                "iscrowd": int(source_annotation.get("iscrowd", 0)),
            })
            box_count += 1

        split_manifest_rows.append({
            "split": split_name,
            "file_name": output_name,
            "source_key": record["source_key"],
            "group_id": record["group_id"],
            "original_split": record["source_split"],
            "boxes": len(record["annotations"]),
        })

    coco_out = {
        "info": {
            "description": "SIRAP USA: RT-DETRv2-S independent grouped split",
            "source_dataset": "European + USA License plates, Roboflow v1",
            "grouping": "manual group_id where provided; otherwise Roboflow source filename",
        },
        "licenses": source_licenses,
        "images": images_out,
        "annotations": annotations_out,
        "categories": [{"id": 1, "name": "license_plate", "supercategory": "license_plate"}],
    }
    (split_folder / "_annotations.coco.json").write_text(
        json.dumps(coco_out, indent=2), encoding="utf-8"
    )
    new_split_summary[split_name] = {
        "images": len(images_out),
        "boxes": box_count,
        "empty_images": sum(1 for image in images_out if not any(a["image_id"] == image["id"] for a in annotations_out)),
        "groups": len({record["group_id"] for record in records}),
    }

with (INDEPENDENT_ROOT / "split_manifest.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["split", "file_name", "source_key", "group_id", "original_split", "boxes"]
    )
    writer.writeheader()
    writer.writerows(split_manifest_rows)

SPLIT_SUMMARY = {
    "dataset": "European + USA License plates",
    "roboflow_version": 1,
    "seed": SPLIT_SEED,
    "validation_group_fraction": VALIDATION_GROUP_FRACTION,
    "grouping_level": (
        "manual group_id plus source filename fallback"
        if manual_groups else "source filename before .rf.; vehicle/session disjointness not verified"
    ),
    "manual_group_ids_filled": len(manual_groups),
    "source_images_before": {
        name: len(records) for name, records in source_records.items()
    },
    "heldout_test_definition": "original Roboflow valid + test",
    "train_images_removed_for_heldout_group_overlap": len(train_overlap),
    "overlapping_train_groups_removed": len({record["group_id"] for record in train_overlap}),
    "sha256_files_audited": hash_count,
    "new_splits": new_split_summary,
}
( INDEPENDENT_ROOT / "split_summary.json" ).write_text(
    json.dumps(SPLIT_SUMMARY, indent=2, ensure_ascii=False), encoding="utf-8"
)
marker.write_text(json.dumps({"generated_by": "SIRAP USA grouped COCO split"}, indent=2), encoding="utf-8")

DATA_DIR = str(INDEPENDENT_ROOT)
INDEPENDENT_TEST_READY = True
print(json.dumps(SPLIT_SUMMARY, indent=2, ensure_ascii=False))
print("Nuevo DATA_DIR para entrenamiento y evaluación:", DATA_DIR)
print("Plantilla opcional para agrupar vehículos/sesiones:", GROUP_MANIFEST)


## Evaluación final del test

La celda siguiente queda apagada. Actívala una sola vez cuando hayas elegido el checkpoint y congelado los hiperparámetros con la nueva validación. Usa el test ampliado únicamente para el informe final; el umbral 0.001 se usa para métricas COCO del detector, no como umbral operativo de acceso.

In [ ]:
from src.evaluate import evaluate_detector

RUN_FINAL_TEST_EVALUATION = False
FINAL_OPENVINO_CHECKPOINT = REPO_ROOT / "models/usa/detector/rtdetrv2_s/openvino"
FINAL_TEST_METRICS_PATH = REPO_ROOT / "results/metrics/rtdetr_usa_independent_test.json"

if RUN_FINAL_TEST_EVALUATION:
    if not globals().get("INDEPENDENT_TEST_READY", False):
        raise RuntimeError("Primero crea y valida el split independiente.")
    if SPLIT_SUMMARY["new_splits"]["test"]["images"] < 400:
        raise RuntimeError("El test es menor de 400 imágenes; no se reportará como ampliado.")
    final_test_metrics = evaluate_detector(
        data_dir=DATA_DIR,
        country="usa",
        split="test",
        checkpoint=FINAL_OPENVINO_CHECKPOINT,
        model_id=MODEL_ID,
        threshold=0.001,
        result_path=FINAL_TEST_METRICS_PATH,
    )
    print(final_test_metrics)
else:
    print(
        "Evaluación final apagada. Primero entrena y congela el modelo; "
        "después activa RUN_FINAL_TEST_EVALUATION una sola vez."
    )


In [ ]:
# Conversión opcional desde anotaciones YOLO previas, si no hay fuente COCO.
DATA_YAML=None  # ejemplo: /content/dataset/data.yaml
if DATA_YAML:
    DATA_DIR=str(convert_yolo_to_coco(DATA_YAML,"/content/dataset-coco"))
    print("Dataset convertido:",DATA_DIR)
